<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_6_KMEANS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="muted")
RANDOM_SEED = 42

# load Wine dataset: 178 samples with 13 continuous chemical measurements across 3 cultivars
wine_bunch = load_wine()
feature_matrix = wine_bunch.data
true_labels = wine_bunch.target
feature_names = wine_bunch.feature_names

wine_df = pd.DataFrame(feature_matrix, columns=feature_names)
wine_df["variety"] = true_labels

# scale features using StandardScaler
# K-Means relies on Euclidean distance, so features with larger ranges (like proline) would dominate without scaling
scaler = StandardScaler()
scaled_features = scaler.fit_transform(feature_matrix)

print(f"Dataset shape: {wine_df.shape}")
wine_df.head(8)

In [ ]:
# determine optimal k using two complementary techniques:
# 1. Elbow Method: looks for an 'elbow' where reduction in WCSS (Within-Cluster Sum of Squares) slows down
# 2. Silhouette Score: measures cluster separation and cohesion (values closer to +1 mean well-separated clusters)
k_range = range(2, 11)
inertia_values = []
sil_scores = []

for k in k_range:
    model = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    preds = model.fit_predict(scaled_features)
    inertia_values.append(model.inertia_)
    sil_scores.append(silhouette_score(scaled_features, preds))

# plot Elbow and Silhouette curves side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inertia_values, marker="o", color="steelblue")
axes[0].set_title("Elbow Method — Inertia (WCSS) vs k")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_range), sil_scores, marker="s", color="coral")
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"Optimal k suggested by silhouette score: {best_k}")

# fit final model with k=3 (matches the 3 true wine cultivar classes)
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=RANDOM_SEED, n_init=10)
wine_df["Cluster"] = kmeans_model.fit_predict(scaled_features)

print(f"Silhouette Score (k={optimal_k}): {silhouette_score(scaled_features, wine_df['Cluster']):.4f}")
print(f"Final Inertia: {kmeans_model.inertia_:.4f}")

# inverse-transform cluster centers back to original feature scale for intuitive interpretation
centroids_original = scaler.inverse_transform(kmeans_model.cluster_centers_)
centroid_df = pd.DataFrame(centroids_original, columns=feature_names)
centroid_df.index.name = "Cluster"
print("
Cluster Centroids in Original Units:")
display(centroid_df)

In [ ]:
print("Clustered sample with true variety:")
display(wine_df[["alcohol", "flavanoids", "color_intensity", "variety", "Cluster"]].head(10))

print("
Sample counts per cluster:")
print(wine_df["Cluster"].value_counts().sort_index())

# scatter plot showing clusters along two strong discriminative features: Alcohol vs Flavanoids
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=wine_df, x="alcohol", y="flavanoids",
    hue="Cluster", palette="Set2", s=80, edgecolor="black", ax=ax
)

# overlay cluster centers marked with red X
ax.scatter(
    centroid_df["alcohol"], centroid_df["flavanoids"],
    marker="X", s=250, c="red", zorder=5, label="Centroids"
)
ax.set_title("K-Means Clustering on Wine (Alcohol vs Flavanoids)", fontsize=13)
ax.set_xlabel("Alcohol Content (% vol)")
ax.set_ylabel("Flavanoids")
ax.legend(title="Cluster")
plt.tight_layout()
plt.show()